# Демо 3. SFT: від повного донавчання до LoRA і QLoRA

У цьому ноутбуці донавчаємо моделі **Qwen3** трьох розмірів виконувати одну вузьку прикладну задачу:
**класифікувати звернення клієнтів служби підтримки** й відповідати лише JSON із категорією та наміром звернення.

### Навіщо потрібен SFT

Інтернет-магазин щодня отримує тисячі повідомлень від клієнтів: скасувати замовлення, змінити адресу доставки,
повернути кошти тощо. Щоб автоматично передавати кожне звернення у відповідний відділ, програмі потрібна відповідь
у строго заданому форматі, наприклад `{"category": "ORDER", "intent": "cancel_order"}`, з мітками з переліку компанії.

Загальна instruct-модель може виконати таку задачу, якщо описати її в промпті, але:
- вона не знає переліку міток компанії, тому його доводиться передавати в кожному запиті;
- навіть із переліком вона часто плутає близькі мітки, бо не знає, як саме компанія їх розрізняє
  (наприклад, `get_refund` і `track_refund`).

Донавчання на прикладах із правильними мітками закріплює формат відповіді й значення кожної мітки у вагах моделі,
тому після нього достатньо короткого промпту.

План:
1. Підготувати дані у форматі `messages` і розбити їх на тренувальний, валідаційний і тестовий набори.
2. Підготувати функції для генерації й оцінювання.
3. Оцінити, чому повне донавчання великої моделі не вміщується в GPU T4.
4. Налаштувати **LoRA** і параметри тренування.
5. Донавчити методом LoRA моделі **Qwen3-0.6B, Qwen3-1.7B і Qwen3-4B** і порівняти, чи може менша модель після донавчання наздогнати більшу.
6. Виконати **повне донавчання** найменшої моделі й порівняти його з LoRA.
7. Зберегти адаптер, завантажити його знову й влити у ваги моделі.
8. Донавчити Qwen3-4B методом **QLoRA** і порівняти витрати памʼяті.

Приклад за мотивами розділу 11.4 [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/4)
і вправ курсу [smol-course](https://huggingface.co/learn/smol-course).
Для запуску в Colab оберіть середовище з GPU.

### Службова клітинка: таймер виконання

Ця клітинка потрібна для відповіді на питання про тривалість виконання: вона підключає до IPython обробник, який після кожної клітинки записує, скільки секунд вона виконувалася. Запускайте її **першою**.

In [1]:
# Службова клітинка до питання про тривалість: запамʼятовує, скільки секунд виконувалася кожна клітинка.
# Запускайте її першою, до всіх інших клітинок ноутбука.
import json
import os
import time

from IPython import get_ipython

TIMES_FILE = "cell_times_03.json"  # файл переживе перезапуск середовища в межах сесії Colab
CELL_TIMES = json.load(open(TIMES_FILE)) if os.path.exists(TIMES_FILE) else []
_started = [None]


def _pre_run(*args):
    _started[0] = time.perf_counter()


def _post_run(result=None):
    if _started[0] is None:  # саму службову клітинку не вимірюємо
        return
    raw = (getattr(getattr(result, "info", None), "raw_cell", "") or "").strip()
    CELL_TIMES.append({
        "cell": getattr(result, "execution_count", None),
        "sec": round(time.perf_counter() - _started[0], 1),
        "code": raw.splitlines()[0][:70] if raw else "",
    })
    with open(TIMES_FILE, "w") as f:
        json.dump(CELL_TIMES, f)


ip = get_ipython()
for event, callback in (("pre_run_cell", _pre_run), ("post_run_cell", _post_run)):
    for old in list(ip.events.callbacks[event]):
        if getattr(old, "__name__", "") == callback.__name__:
            ip.events.unregister(event, old)
    ip.events.register(event, callback)
print("Таймер клітинок увімкнено")

Таймер клітинок увімкнено


In [2]:
# trl — бібліотека для донавчання мовних моделей (SFT, DPO тощо),
# peft — реалізація LoRA та інших методів PEFT, bitsandbytes — 4-бітна квантизація для QLoRA
!pip install -q "transformers>=4.55.0" "trl>=1.17" "peft" "bitsandbytes" "datasets" "accelerate"
# Colab має застарілу бібліотеку torchao, з якою peft не імпортується; для цього демо вона не потрібна
!pip uninstall -q -y torchao

In [3]:
import gc
import json
import os
import time

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

assert torch.cuda.is_available(), "Для цього ноутбука потрібна GPU"
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla T4
Памʼять GPU: 15.6 GB


## 1. Дані

[Bitext Customer Support](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset) —
датасет із майже 27 тис. повідомлень клієнтів англійською мовою. Кожне повідомлення має дві мітки:
- `category` — відділ, якого стосується звернення (11 категорій, наприклад `ORDER`, `REFUND`, `ACCOUNT`);
- `intent` — конкретний намір клієнта (27 намірів, наприклад `cancel_order`, `track_refund`).

Конкретні дані в повідомленнях замінено заповнювачами на кшталт `{{Order Number}}`.

In [4]:
raw_dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
print(raw_dataset)

for ex in raw_dataset.shuffle(seed=0).select(range(5)):
    print(f"{ex['category']:<10} {ex['intent']:<25} {ex['instruction']}")

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response'],
    num_rows: 26872
})
SHIPPING   set_up_shipping_address   there are errors setting up a delivery address
PAYMENT    check_payment_methods     i want assistance seeing ur accepted payment opotions
SUBSCRIPTION newsletter_subscription   I have got to cancel the newsletter subscription
ACCOUNT    switch_account            can ya help me to change to the gold account
REFUND     track_refund              I am waiting for a compensation of {{Currency Symbol}}{{Refund Amount}}


In [5]:
categories = sorted(set(raw_dataset["category"]))
intents = sorted(set(raw_dataset["intent"]))
print(f"Категорій: {len(categories)}, намірів: {len(intents)}")
print(categories)

Категорій: 11, намірів: 27
['ACCOUNT', 'CANCEL', 'CONTACT', 'DELIVERY', 'FEEDBACK', 'INVOICE', 'ORDER', 'PAYMENT', 'REFUND', 'SHIPPING', 'SUBSCRIPTION']


### Дедуплікація

Частина звернень у датасеті повторюється дослівно. Якщо такий дублікат потрапить і в тренувальний, і у валідаційний набір,
validation loss буде заниженим: модель перевірятимуть на прикладах, які вона вже бачила.
Тому перед розбиттям залишаємо по одному прикладу для кожного тексту звернення (без урахування регістру й пробілів на краях).

In [6]:
seen, keep = set(), []
for i, text in enumerate(raw_dataset["instruction"]):
    key = text.strip().lower()
    if key not in seen:
        seen.add(key)
        keep.append(i)

dedup_dataset = raw_dataset.select(keep)
print(f"До дедуплікації: {len(raw_dataset)}, після: {len(dedup_dataset)}")

До дедуплікації: 26872, після: 24371


### Формат `messages`

Кожне звернення перетворюємо на діалог із трьох повідомлень:
- `system` — коротка інструкція;
- `user` — повідомлення клієнта;
- `assistant` — правильна відповідь: JSON із мітками.

Колонка `chat_template_kwargs` вимикає режим міркувань Qwen3: модель має одразу відповідати JSON.

In [7]:
SHORT_PROMPT = "Classify the customer message. Reply with JSON only."


def to_messages(example):
    answer = json.dumps({"category": example["category"], "intent": example["intent"]})
    return {
        "messages": [
            {"role": "system", "content": SHORT_PROMPT},
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": answer},
        ],
        "chat_template_kwargs": {"enable_thinking": False},
    }

### Тренувальний, валідаційний і тестовий набори

- **Тренувальний** (2800 прикладів) — на ньому модель вчиться.
- **Валідаційний** (200) — на ньому під час тренування обчислюється validation loss, щоб помітити перенавчання.
- **Тестовий** (1000) — на ньому лише один раз, після тренування, рахуємо підсумкові метрики.
  Під час тренування й налаштування ці приклади не використовуються, тому оцінка на них найчесніша.

Після дедуплікації та перемішування тестовий набір беремо з прикладів, які не потрапили до перших 3000.

Тестовий набір більший за валідаційний, бо від його розміру залежить точність вимірювання метрик:
на 1000 прикладах одна помилка змінює точність на 0,1%, тоді як на 200 — на 0,5%.
Донавчені моделі помиляються рідко, і на малому тестовому наборі різниця між ними може бути непомітною.

In [8]:
shuffled = dedup_dataset.shuffle(seed=42)

dataset = shuffled.select(range(3000)).map(to_messages, remove_columns=shuffled.column_names)
dataset = dataset.train_test_split(test_size=200, seed=42)
train_dataset, val_dataset = dataset["train"], dataset["test"]
test_dataset = shuffled.select(range(3000, 4000)).map(to_messages, remove_columns=shuffled.column_names)

print(f"Тренувальний: {len(train_dataset)}, валідаційний: {len(val_dataset)}, тестовий: {len(test_dataset)}")
train_dataset[0]["messages"]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Тренувальний: 2800, валідаційний: 200, тестовий: 1000


[{'content': 'Classify the customer message. Reply with JSON only.',
  'role': 'system'},
 {'content': 'I have got to locate hte bills from {{Person Name}}',
  'role': 'user'},
 {'content': '{"category": "INVOICE", "intent": "check_invoice"}',
  'role': 'assistant'}]

## 2. Функції для генерації й оцінювання

Порівнюємо три моделі однієї сімʼї з однаковою архітектурою й даними попереднього тренування — **Qwen3** на 0,6, 1,7 і 4 млрд параметрів.
Усі вони мають той самий токенізатор і шаблон розмови, тому токенізатор завантажуємо один раз.

In [9]:
MODELS = ["Qwen/Qwen3-0.6B", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-4B"]

tokenizer = AutoTokenizer.from_pretrained(MODELS[0])

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

### Детальний промпт

Без донавчання модель не знає переліку міток, тому для оцінки «до» передаємо його в промпті разом з описом формату.

In [10]:
LONG_PROMPT = f"""Classify the customer message for the support ticket routing system.
Reply with a single JSON object and nothing else, in the format:
{{"category": "<CATEGORY>", "intent": "<intent>"}}

Allowed categories: {", ".join(categories)}
Allowed intents: {", ".join(intents)}"""

for name, prompt in [("Короткий промпт", SHORT_PROMPT), ("Детальний промпт", LONG_PROMPT)]:
    print(f"{name}: {len(tokenizer(prompt)['input_ids'])} токенів")

Короткий промпт: 11 токенів
Детальний промпт: 166 токенів


### Метрики

Відповідь моделі вважаємо правильною, якщо програма може розібрати її як JSON і мітки збігаються з еталонними.
До донавчання моделі Qwen3 часто загортають JSON у блок розмітки Markdown (` ```json … ``` `), тому таку обгортку перед розбором знімаємо,
але окремо рахуємо, як часто відповідь є чистим JSON без зайвого тексту.

Рахуємо чотири метрики на всьому тестовому наборі:
- частку відповідей, які є чистим JSON;
- частку відповідей, які після зняття обгортки є валідним JSON з полями `category` і `intent`;
- точність категорії;
- точність наміру.

Щоб оцінка 1000 прикладів не тривала надто довго, відповіді генеруються батчами по 32 звернення.

In [11]:
def classify_batch(model, messages_batch, system_prompt, max_new_tokens=60):
    """Повертає відповіді моделі на список звернень клієнтів, згенеровані одним батчем."""
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": system_prompt}, {"role": "user", "content": message}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        for message in messages_batch
    ]
    # Для генерації батчем короткі промпти доповнюються зліва, щоб відповіді всіх промптів починалися в одній позиції
    tokenizer.padding_side = "left"
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    tokenizer.padding_side = "right"
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id
        )
    new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    return [answer.strip() for answer in tokenizer.batch_decode(new_tokens, skip_special_tokens=True)]


def classify(model, message, system_prompt):
    """Повертає відповідь моделі на одне звернення клієнта."""
    return classify_batch(model, [message], system_prompt)[0]


def is_clean_json(answer):
    """Чи є відповідь JSON без жодного зайвого тексту."""
    try:
        json.loads(answer)
        return True
    except json.JSONDecodeError:
        return False


def parse(answer):
    """Знімає обгортку ```json … ```, розбирає JSON; повертає None, якщо формат неправильний."""
    text = answer.strip()
    if text.startswith("```"):
        text = text.strip("`").removeprefix("json").strip()
    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        return None
    if isinstance(result, dict) and {"category", "intent"} <= result.keys():
        return result
    return None


def evaluate(model, system_prompt, batch_size=32):
    """Рахує метрики на всьому тестовому наборі."""
    model.eval()
    messages = [ex["messages"][1]["content"] for ex in test_dataset]
    targets = [json.loads(ex["messages"][2]["content"]) for ex in test_dataset]
    answers = []
    for start in range(0, len(messages), batch_size):
        answers += classify_batch(model, messages[start:start + batch_size], system_prompt)

    clean = valid = category_ok = intent_ok = 0
    for answer, target in zip(answers, targets):
        clean += is_clean_json(answer)
        parsed = parse(answer)
        if parsed is not None:
            valid += 1
            category_ok += parsed["category"] == target["category"]
            intent_ok += parsed["intent"] == target["intent"]
    n = len(answers)
    metrics = {
        "Чистий JSON, %": 100 * clean / n,
        "Валідний JSON, %": 100 * valid / n,
        "Точність category, %": 100 * category_ok / n,
        "Точність intent, %": 100 * intent_ok / n,
    }
    return metrics, list(zip(messages, answers))


def show(answers, k=5):
    """Друкує перші k пар «звернення — відповідь моделі»."""
    for message, answer in answers[:k]:
        print(f"📩 {message}\n🤖 {answer}\n" + "-" * 60)


def free_memory():
    """Звільняє памʼять GPU після видалення моделей."""
    gc.collect()
    torch.cuda.empty_cache()

## 3. Чому не повне донавчання

Під час повного донавчання з оптимізатором Adam у змішаній точності на кожен параметр потрібно приблизно 16 байт:

| Що зберігається | Байтів на параметр |
|---|---|
| ваги у fp16 | 2 |
| градієнти у fp16 | 2 |
| копія ваг у fp32: у fp16 малі оновлення ваг губляться через округлення | 4 |
| два стани Adam у fp32: усереднені градієнт і його квадрат, за якими Adam підбирає крок для кожного параметра | 8 |
| **разом** | **16** |

До цього додаються активації, які залежать від довжини послідовностей і розміру батча.

Для Qwen3-4B це близько 64 GB без активацій, тоді як T4 має 15 GB: повне донавчання такої моделі потребує GPU рівня A100 на 80 GB.
Для Qwen3-0.6B потрібно лише близько 10 GB, тому в розділі 6 виконаємо повне донавчання саме цієї моделі.

## 4. Налаштування LoRA і тренування

LoRA заморожує всі ваги моделі й додає до вибраних шарів дві невеликі матриці $A$ і $B$.
Тренуються лише вони, тому градієнти й стани оптимізатора потрібні тільки для них.

Параметри `LoraConfig`:
- `r` — ранг матриць адаптера: що більший, то більше параметрів і «свободи» має адаптер;
- `lora_alpha` — масштаб внеску адаптера (зазвичай `2 × r`);
- `lora_dropout` — dropout в адаптері для регуляризації;
- `target_modules="all-linear"` — адаптери додаються до всіх лінійних шарів (attention і MLP).

In [12]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

Параметри тренування (`SFTConfig`):
- `assistant_only_loss=True` — loss рахується лише на відповідях асистента, а не на репліках користувача й системному промпті:
  для решти токенів мітки замінюються значенням −100, і функція втрат їх пропускає;
- `max_steps=100` — кількість кроків тренування; за один крок модель бачить 8 прикладів, тобто за все тренування — 800 із 2800,
  менше третини епохи. Для вузької задачі цього достатньо: validation loss виходить на плато раніше;
- `learning_rate=2e-4` — для LoRA learning rate приблизно в 10 разів більший, ніж для повного донавчання;
- `eval_strategy="steps"` і `eval_steps=10` — кожні 10 кроків модель оцінюється на валідаційному наборі:
  якщо validation loss почне зростати, модель перенавчається;
- `fp16=True` — змішана точність (T4 не підтримує bf16);
- `gradient_checkpointing=True` — активації не зберігаються, а перераховуються під час зворотного проходу: повільніше, але потрібно менше памʼяті.

In [13]:
def make_config(output_dir, learning_rate=2e-4):
    return SFTConfig(
        output_dir=output_dir,
        max_length=256,                  # діалог із цього датасету після шаблону займає близько 100 токенів
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,   # ефективний розмір батча: 4 × 2 = 8 прикладів
        per_device_eval_batch_size=8,
        learning_rate=learning_rate,
        lr_scheduler_type="cosine",
        warmup_steps=5,
        max_steps=100,
        eval_strategy="steps",
        eval_steps=10,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        assistant_only_loss=True,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
    )


def train(model, output_dir, peft_config=peft_config, learning_rate=2e-4):
    """Тренує модель (з LoRA, якщо задано peft_config) і повертає тренер, тривалість, пікову памʼять GPU та validation loss."""
    torch.cuda.reset_peak_memory_stats()
    trainer = SFTTrainer(
        model=model,
        args=make_config(output_dir, learning_rate),
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        peft_config=peft_config,
    )
    # Для QLoRA бібліотека TRL переводить ваги адаптера в bf16, який T4 не підтримує.
    # Повертаємо їх у fp32: змішана точність fp16 потребує, щоб параметри, що тренуються, були у fp32
    if getattr(model, "is_loaded_in_4bit", False):
        for param in trainer.model.parameters():
            if param.requires_grad:
                param.data = param.data.float()
    if peft_config is not None:
        trainer.model.print_trainable_parameters()
    start = time.time()
    trainer.train()
    eval_losses = [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h]
    stats = {
        "Тривалість, хв": (time.time() - start) / 60,
        "Пікова памʼять GPU, GB": torch.cuda.max_memory_allocated() / 1e9,
        "Validation loss": eval_losses[-1],
    }
    return trainer, stats

## 5. LoRA на моделях різного розміру

Для кожної моделі виконуємо однакові кроки:
1. завантажуємо модель у fp16 (ваги Qwen3 збережено в bfloat16, але T4 не підтримує його апаратно);
2. оцінюємо її **до донавчання** з детальним промптом, у якому перелічено всі мітки;
3. донавчаємо методом LoRA з однаковими параметрами;
4. оцінюємо **після донавчання** з коротким промптом без переліку міток;
5. зберігаємо адаптер і вивантажуємо модель з GPU, щоб звільнити памʼять для наступної.

In [14]:
results = {}


def run_lora(model_name):
    """Повний цикл для однієї моделі: оцінка до, LoRA, оцінка після, збереження адаптера."""
    short_name = model_name.split("/")[-1]
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16, device_map="auto")
    print(f"{short_name}: {model.num_parameters() / 1e9:.2f} млрд параметрів, "
          f"памʼять GPU після завантаження: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

    before_metrics, before_answers = evaluate(model, LONG_PROMPT)
    print("\nДо донавчання:")
    show(before_answers, k=3)

    trainer, stats = train(model, f"{short_name}-lora")
    after_metrics, after_answers = evaluate(trainer.model, SHORT_PROMPT)
    print("\nПісля донавчання:")
    show(after_answers, k=3)

    adapter_dir = f"{short_name}-lora-adapter"
    trainer.save_model(adapter_dir)
    stats["Адаптер, MB"] = os.path.getsize(os.path.join(adapter_dir, "adapter_model.safetensors")) / 1e6

    results[short_name] = {"до": before_metrics, "LoRA": after_metrics | stats}
    del trainer, model
    free_memory()
    return pd.DataFrame(results[short_name]).round(2)

In [15]:
run_lora("Qwen/Qwen3-0.6B")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-0.6B: 0.60 млрд параметрів, памʼять GPU після завантаження: 1.2 GB

До донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "CONTACT", "intent": "contact_customer_service"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "CONTACT", "intent": "contact_customer_service"}
------------------------------------------------------------


Tokenizing train dataset:   0%|          | 0/2800 [00:00<?, ? examples/s]

... (7 outputs left out)

trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.362180,0.478099,0.437587,0.894284,4323.000000
20,0.321516,0.142696,0.187930,0.956261,8672.000000
30,0.081163,0.061503,0.072797,0.985931,13032.000000
40,0.059188,0.027427,0.045873,0.990944,17377.000000
50,0.025087,0.011763,0.028740,0.996557,21714.000000
60,0.006312,0.008056,0.018671,0.998110,26070.000000
70,0.011715,0.006804,0.016659,0.998124,30452.000000
80,0.007442,0.005719,0.013954,0.998745,34813.000000
90,0.005135,0.005946,0.013778,0.998755,39116.000000
100,0.016230,0.005854,0.013654,0.998755,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",37.9,99.70
"Точність intent, %",25.6,97.70
"Тривалість, хв",NaN,2.66
"Пікова памʼять GPU, GB",NaN,1.42
Validation loss,NaN,0.01
"Адаптер, MB",NaN,40.42


In [16]:
run_lora("Qwen/Qwen3-1.7B")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (2 outputs left out)

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.993199,0.438076,0.369669,0.905950,4323.000000
20,0.315701,0.133664,0.192973,0.955675,8672.000000
30,0.080244,0.045514,0.050354,0.987794,13032.000000
40,0.049900,0.024616,0.028389,0.990317,17377.000000
50,0.017411,0.017447,0.025503,0.994078,21714.000000
60,0.012290,0.005510,0.014741,0.998740,26070.000000
70,0.012933,0.005394,0.011520,0.997189,30452.000000
80,0.011438,0.005137,0.010611,0.997494,34813.000000
90,0.013357,0.004242,0.010540,0.997812,39116.000000
100,0.009885,0.004055,0.010473,0.997812,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",62.7,99.30
"Точність intent, %",57.4,96.60
"Тривалість, хв",NaN,2.70
"Пікова памʼять GPU, GB",NaN,3.83
Validation loss,NaN,0.00
"Адаптер, MB",NaN,69.78


In [17]:
run_lora("Qwen/Qwen3-4B")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (2 outputs left out)

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.768687,0.299795,0.305966,0.923112,4323.000000
20,0.202206,0.100573,0.100039,0.962436,8672.000000
30,0.065379,0.029711,0.035359,0.989987,13032.000000
40,0.035180,0.018227,0.026112,0.995031,17377.000000
50,0.013948,0.007081,0.015536,0.998422,21714.000000
60,0.009037,0.003752,0.009470,0.998437,26070.000000
70,0.007353,0.005263,0.009148,0.998117,30452.000000
80,0.012355,0.004769,0.008412,0.998110,34813.000000
90,0.015855,0.003387,0.008266,0.998742,39116.000000
100,0.012198,0.003268,0.008148,0.998742,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",88.1,100.00
"Точність intent, %",72.5,97.80
"Тривалість, хв",NaN,3.93
"Пікова памʼять GPU, GB",NaN,8.75
Validation loss,NaN,0.00
"Адаптер, MB",NaN,132.19


### Порівняння

Чи може менша модель після донавчання досягти якості більшої? Порівняйте точність, тривалість тренування, памʼять і розмір адаптера.

In [18]:
pd.DataFrame({
    f"{name} {stage}": metrics
    for name, stages in results.items()
    for stage, metrics in stages.items()
}).round(2)

,Qwen3-0.6B до,Qwen3-0.6B LoRA,Qwen3-1.7B до,Qwen3-1.7B LoRA,Qwen3-4B до,Qwen3-4B LoRA
"Чистий JSON, %",100.0,100.00,100.0,100.00,100.0,100.00
"Валідний JSON, %",100.0,100.00,100.0,100.00,100.0,100.00
"Точність category, %",37.9,99.70,62.7,99.30,88.1,100.00
"Точність intent, %",25.6,97.70,57.4,96.60,72.5,97.80
"Тривалість, хв",NaN,2.66,NaN,2.70,NaN,3.93
"Пікова памʼять GPU, GB",NaN,1.42,NaN,3.83,NaN,8.75
Validation loss,NaN,0.01,NaN,0.00,NaN,0.00
"Адаптер, MB",NaN,40.42,NaN,69.78,NaN,132.19


## 6. Повне донавчання Qwen3-0.6B

Модель на 0,6 млрд параметрів потребує для повного донавчання близько 10 GB, тому на T4 її можна донавчити повністю й порівняти з LoRA.

Відмінності від LoRA:
- `peft_config=None` — тренуються всі ваги моделі;
- модель завантажуємо у **fp32**: змішана точність fp16 тримає головну копію ваг у fp32, а в fp16 лише обчислює прохід;
- `learning_rate=2e-5` — для повного донавчання learning rate приблизно в 10 разів менший, ніж для LoRA.

In [19]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", dtype=torch.float32, device_map="auto")
trainer, full_stats = train(model, "Qwen3-0.6B-full", peft_config=None, learning_rate=2e-5)
full_metrics, full_answers = evaluate(trainer.model, SHORT_PROMPT)
show(full_answers, k=3)

del trainer, model
free_memory()

pd.DataFrame({"Qwen3-0.6B LoRA": results["Qwen3-0.6B"]["LoRA"], "Qwen3-0.6B повне": full_metrics | full_stats}).round(2)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.255744,0.288718,0.251618,0.934379,4323.000000
20,0.176103,0.060587,0.091798,0.985634,8672.000000
30,0.049239,0.038266,0.028003,0.990942,13032.000000
40,0.046789,0.025577,0.022165,0.994027,17377.000000
50,0.026981,0.007719,0.010975,0.998120,21714.000000
60,0.002136,0.004413,0.004073,0.998442,26070.000000
70,0.005817,0.004062,0.003158,0.999375,30452.000000
80,0.012955,0.003829,0.004107,0.998437,34813.000000
90,0.009278,0.003025,0.003515,0.999370,39116.000000
100,0.014485,0.002765,0.003435,0.999370,43449.000000


📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,Qwen3-0.6B LoRA,Qwen3-0.6B повне
"Чистий JSON, %",100.00,100.00
"Валідний JSON, %",100.00,100.00
"Точність category, %",99.70,99.50
"Точність intent, %",97.70,98.30
"Тривалість, хв",2.66,1.78
"Пікова памʼять GPU, GB",1.42,10.80
Validation loss,0.01,0.00
"Адаптер, MB",40.42,NaN


## 7. Збереження і завантаження адаптера

`save_model` зберігає **лише адаптер** (`adapter_model.safetensors`) разом із його налаштуваннями й токенізатором, а не всю модель.

In [20]:
adapter_dir = "Qwen3-4B-lora-adapter"
for name in sorted(os.listdir(adapter_dir)):
    size = os.path.getsize(os.path.join(adapter_dir, name))
    print(f"{name:<35} {size / 1e6:>8.1f} MB")
print("\nДля порівняння, ваги Qwen3-4B у fp16: ~8 GB")

README.md                                0.0 MB
adapter_config.json                      0.0 MB
adapter_model.safetensors              132.2 MB
chat_template.jinja                      0.0 MB
tokenizer.json                          11.4 MB
tokenizer_config.json                    0.0 MB
training_args.bin                        0.0 MB

Для порівняння, ваги Qwen3-4B у fp16: ~8 GB


Щоб використати адаптер, завантажуємо модель і накладаємо на неї адаптер з диска.
Метод `merge_and_unload` вливає адаптер у ваги моделі: $W' = W + \frac{\alpha}{r} BA$.
Після цього модель працює як звичайна, без додаткових обчислень для адаптера.

In [21]:
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, adapter_dir)
model = model.merge_and_unload()
print(type(model).__name__)

# Злиту модель можна зберегти як звичайну: model.save_pretrained("qwen3-4b-merged")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForCausalLM


Перевіримо злиту модель на власних повідомленнях:

In [22]:
my_messages = [
    "Hi, I ordered a laptop last week and it still hasn't arrived. Where is it?",
    "I was charged twice for the same order, please help",
    "How can I talk to a real person?",
    "I guess, you have shipped to a wrong address",
]
for message in my_messages:
    print(f"📩 {message}\n🤖 {classify(model, message, SHORT_PROMPT)}\n")

📩 Hi, I ordered a laptop last week and it still hasn't arrived. Where is it?
🤖 {"category": "DELIVERY", "intent": "delivery_period"}

📩 I was charged twice for the same order, please help
🤖 {"category": "ORDER", "intent": "cancel_order"}

📩 How can I talk to a real person?
🤖 {"category": "CONTACT", "intent": "contact_human_agent"}

📩 I guess, you have shipped to a wrong address
🤖 {"category": "SHIPPING", "intent": "shipping_error"}



In [23]:
del model
free_memory()

## 8. QLoRA

QLoRA завантажує заморожені ваги Qwen3-4B в **4 бітах** (формат NF4), а адаптери LoRA тренує у звичайній точності.
Код тренування той самий, змінюється лише завантаження моделі.

### Формат NF4

У 4 бітах вага може набувати лише одного з 16 значень. Ваги ділять на блоки по 64, і для кожного блоку
зберігають масштаб — найбільше за модулем значення в блоці. Формати відрізняються тим, як розташовані ці 16 значень:
- **INT4** — 16 рівновіддалених значень. Ваги нейромережі розподілені приблизно нормально: більшість близька до нуля,
  а великих за модулем мало. Тому крайні значення використовуються рідко, а біля нуля їх бракує;
- **NF4 (NormalFloat4)** — 16 значень розташовані за квантилями нормального розподілу: густо біля нуля й рідко на краях.
  Кожне значення використовується приблизно однаково часто, тому помилка квантизації менша, ніж у INT4.

NF4 — формат лише для зберігання: перед кожним множенням ваги деквантизуються у fp16 (`bnb_4bit_compute_dtype`).
Детальніше — у блозі Hugging Face [Making LLMs even more accessible with bitsandbytes, 4-bit quantization and QLoRA](https://huggingface.co/blog/4bit-transformers-bitsandbytes)
і в статті [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314).

In [24]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # 4-бітний формат, розрахований на нормально розподілені ваги
    bnb_4bit_compute_dtype=torch.float16,   # обчислення виконуються у fp16
    bnb_4bit_use_double_quant=True,         # квантизуються також масштаби квантизації: ще трохи менше памʼяті
)

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", quantization_config=bnb_config, device_map="auto")
print(f"Памʼять GPU після завантаження: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Памʼять GPU після завантаження: 2.7 GB


In [25]:
trainer, qlora_stats = train(model, "Qwen3-4B-qlora")
qlora_metrics, _ = evaluate(trainer.model, SHORT_PROMPT)
qlora_stats

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.482167,0.315638,0.297774,0.923437,4323.000000
20,0.196171,0.127783,0.097346,0.957768,8672.000000
30,0.071772,0.025973,0.042697,0.993133,13032.000000
40,0.029052,0.014744,0.022618,0.996564,17377.000000
50,0.015930,0.012615,0.016141,0.995899,21714.000000
60,0.008367,0.005350,0.008351,0.998728,26070.000000
70,0.006580,0.006176,0.007877,0.998110,30452.000000
80,0.003290,0.006270,0.007162,0.997485,34813.000000
90,0.013268,0.004991,0.007232,0.998422,39116.000000
100,0.011323,0.004792,0.007254,0.998422,43449.000000


{'Тривалість, хв': 5.128996280829112,
 'Пікова памʼять GPU, GB': 3.665665536,
 'Validation loss': 0.004792219027876854}

## 9. Підсумкове порівняння

In [26]:
summary = {
    f"{name} {stage}": metrics
    for name, stages in results.items()
    for stage, metrics in stages.items()
}
summary["Qwen3-0.6B повне"] = full_metrics | full_stats
summary["Qwen3-4B QLoRA"] = qlora_metrics | qlora_stats
pd.DataFrame(summary).round(2)

,Qwen3-0.6B до,Qwen3-0.6B LoRA,Qwen3-1.7B до,Qwen3-1.7B LoRA,Qwen3-4B до,Qwen3-4B LoRA,Qwen3-0.6B повне,Qwen3-4B QLoRA
"Чистий JSON, %",100.0,100.00,100.0,100.00,100.0,100.00,100.00,100.00
"Валідний JSON, %",100.0,100.00,100.0,100.00,100.0,100.00,100.00,100.00
"Точність category, %",37.9,99.70,62.7,99.30,88.1,100.00,99.50,99.90
"Точність intent, %",25.6,97.70,57.4,96.60,72.5,97.80,98.30,97.80
"Тривалість, хв",NaN,2.66,NaN,2.70,NaN,3.93,1.78,5.13
"Пікова памʼять GPU, GB",NaN,1.42,NaN,3.83,NaN,8.75,10.80,3.67
Validation loss,NaN,0.01,NaN,0.00,NaN,0.00,0.00,0.00
"Адаптер, MB",NaN,40.42,NaN,69.78,NaN,132.19,NaN,NaN


**Що варто помітити:**
- LoRA тренує близько 1–2% параметрів моделі, а адаптер займає десятки або сотні мегабайт замість гігабайтів для всієї моделі.
- Порівняйте моделі різного розміру: наскільки менша модель після донавчання поступається більшій і скільки вона заощаджує часу й памʼяті.
  Для вузької задачі з тисячами прикладів донавчена мала модель може бути вигіднішою за велику.
- Порівняйте повне донавчання Qwen3-0.6B з LoRA на тій самій моделі за якістю, часом і памʼяттю.
- До донавчання моделі отримують детальний промпт із переліком міток, після — лише короткий:
  формат і значення міток тепер зберігаються у вагах, а не в промпті. Коротший промпт означає швидшу й дешевшу обробку кожного запиту.
- QLoRA потребує помітно менше памʼяті, ніж LoRA на тій самій моделі, але тренується повільніше, бо ваги щоразу деквантизуються під час обчислень.

## Відповіді на теоретичні питання (0.1)

Питання 22–30 до цього ноутбука. До відповідей додано код, який їх підтверджує. Клітинки нижче потрібно запускати **після** основної частини ноутбука (усіх розділів 1–9): вони використовують `train_dataset`, `test_dataset`, `tokenizer`, `summary`, `parse`, `classify_batch` та інші змінні, створені вище.

### Підготовка: звільняємо GPU

Наприкінці демо на GPU лишається QLoRA-модель разом із тренером. Для експериментів нижче потрібна вільна памʼять, тож видаляємо їх (вони більше не знадобляться).

In [27]:
for name in ("trainer", "model"):
    globals().pop(name, None)
free_memory()
print(f"Памʼять GPU після очищення: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Памʼять GPU після очищення: 0.02 GB


### Питання 22. Як довго виконувалися клітинки ноутбуку 03_sft_lora? Чи були проблеми з запуском, які?

**Відповідь.** Час виконання вимірювала службова клітинка на початку ноутбука: вона записує тривалість кожної клітинки, а підсумок виведено клітинкою нижче.

**Скільки це тривало** (безкоштовний Colab, GPU Tesla T4):
- за таймером, на момент цієї таблиці виконано 26 клітинок, разом **35,2 хв**. Найдовші: `run_lora("Qwen/Qwen3-4B")` — 666,2 с (11,1 хв); тренування й оцінювання QLoRA — 450,6 с (7,5 хв); `run_lora("Qwen/Qwen3-1.7B")` — 352,6 с (5,9 хв); `run_lora("Qwen/Qwen3-0.6B")` — 321,5 с (5,4 хв); повне донавчання Qwen3-0.6B — 159,2 с (2,7 хв); імпорти (`torch`, `trl`, `peft`) — 57,8 с;
- сама тренувальна частина коротша за ці цифри: за таблицею розділу 9 тренування тривало 2,66 хв (0,6B), 2,70 хв (1,7B), 3,93 хв (4B LoRA), 5,13 хв (4B QLoRA) і 1,78 хв (повне донавчання 0,6B). Решту часу займають завантаження моделі, оцінювання до й після донавчання (по 1000 тестових звернень) і збереження адаптера;
- клітинки з відповідями на питання 23–30 виконуються вже після таймера, тому в таблицю не потрапили: найдовше в них завантаження моделей у питанні 23 (три моделі в fp16 і одна в NF4) та чотири прогони генерації в питанні 25.

Разом ноутбук виконується приблизно за **40 хв** (оцінка: 35,2 хв — за таймером, ще ≈ 5 хв — питання 23–30; точнішої цифри таймер не дає, бо його підсумок виведено раніше за ці клітинки).

**Проблеми з запуском.**
1. Помилок виконання (traceback) у запуску немає: усі клітинки пройшли.
2. Попередження, які не є помилками: `You are sending unauthenticated requests to the HF Hub` — завантаження без токена Hugging Face (працює, але з нижчим лімітом запитів); `The PEFT config's base_model_name_or_path was renamed from 'Qwen/Qwen3-0.6B' to 'Qwen/Qwen3-1.7B'` (і далі до `Qwen3-4B`) під час `run_lora` для другої й третьої моделі. Причина: один об'єкт `peft_config` використовується в усіх трьох запусках, і PEFT щоразу записує в нього назву поточної базової моделі. Це безпечно: кожен адаптер зберігається з назвою тієї моделі, на якій його тренували (адаптери 4B і 0.6B пізніше завантажуються на свої моделі без жодних попереджень). Щоб попередження не зʼявлялося, у `train` можна передавати копію конфігурації (`copy.deepcopy(peft_config)`).
3. Потрібна GPU, і памʼять T4 (15 GB) — на межі для деяких кроків: повне донавчання 0.6B займає 10,8 GB, LoRA на 4B — 8,75 GB. Тому між запусками викликається `free_memory()`, а повне донавчання 4B на T4 неможливе (розділ 3).
4. Ваги Qwen3 і датасет завантажуються з Hugging Face Hub, тож перший запуск залежить від швидкості мережі.

In [28]:
import pandas as pd


def timing_report(top=6):
    """Скільки часу забрали клітинки ноутбука (за даними службової клітинки з початку ноутбука)."""
    times = pd.DataFrame(CELL_TIMES)
    print(f"Виконано клітинок: {len(times)}; сумарний час їх виконання: {times['sec'].sum() / 60:.1f} хв")
    print(f"Найдовші {top} клітинок:")
    return times.sort_values("sec", ascending=False).head(top)[["cell", "sec", "code"]].reset_index(drop=True)


timing_report()

Виконано клітинок: 26; сумарний час їх виконання: 35.2 хв
Найдовші 6 клітинок:


,cell,sec,code
0,17,666.2,"run_lora(""Qwen/Qwen3-4B"")"
1,25,450.6,"trainer, qlora_stats = train(model, ""Qwen3-4B-..."
2,16,352.6,"run_lora(""Qwen/Qwen3-1.7B"")"
3,15,321.5,"run_lora(""Qwen/Qwen3-0.6B"")"
4,19,159.2,"model = AutoModelForCausalLM.from_pretrained(""..."
5,3,57.8,import gc


### Питання 23. Прокоментуйте вивід "Памʼять GPU після завантаження". Чи очікуваним для Вас є результат?

**Відповідь.** Так, результат очікуваний: одразу після завантаження в памʼяті GPU лежать **лише ваги моделі** (ще немає ні активацій, ні градієнтів, ні KV-кешу), тому памʼять ≈ кількість параметрів × байтів на параметр.

- **fp16** — 2 байти на параметр: Qwen3-0.6B ≈ 1,2 GB, Qwen3-1.7B ≈ 3,5 GB, Qwen3-4B ≈ 8 GB (приблизно так, як у виводі `run_lora`). Памʼять зростає майже лінійно з розміром моделі.
- **QLoRA (NF4)** — ваги лінійних шарів зберігаються в 4 бітах, тобто ≈ 0,5 байта на параметр плюс константи квантизації (~3% при подвійній квантизації). Проте виходить не в 4 рази менше, ніж 8 GB: **ембединги** (151 936 токенів × 2560 вимірів, ≈ 0,8 GB) і нормалізації лишаються в fp16, а в 4 біти квантизуються лише лінійні шари. Очікуємо ≈ 2,5–3 GB замість ідеальних 2 GB.

Клітинка нижче рахує очікуване значення за формулою (кількість параметрів визначаємо на «порожній» meta-моделі без завантаження ваг), заново завантажує всі моделі й порівнює з виміряною памʼяттю.

**Результат запуску.** Памʼять після завантаження в fp16: Qwen3-0.6B — 1,21 GB (за формулою «параметри × 2 байти» очікувалось 1,19 GB), Qwen3-1.7B — 3,46 GB (очікувалось 3,44), Qwen3-4B — 8,06 GB (очікувалось 8,04). Відхилення не більше 1,5%, тобто результат очікуваний: у памʼяті лише ваги. У форматі NF4 Qwen3-4B займає 2,70 GB (за формулою 2,65), тобто в **3,0 раза** менше, ніж у fp16 (8,06 GB), а не в 4: ембединги й нормалізації лишаються в fp16. Приблизно так само показав і вивід `run_lora`: 1,2 GB, 3,5 GB і 8,1 GB.

In [29]:
from transformers import AutoConfig


def count_parameters(model_name):
    """Кількість параметрів і частка лінійних шарів без завантаження ваг: модель створюється на meta-пристрої."""
    config = AutoConfig.from_pretrained(model_name)
    with torch.device("meta"):
        meta_model = AutoModelForCausalLM.from_config(config)
    total = sum(p.numel() for p in meta_model.parameters())  # звʼязані ваги (ембединги й lm_head) рахуються один раз
    linear = sum(m.weight.numel() for n, m in meta_model.named_modules() if isinstance(m, torch.nn.Linear) and n != "lm_head")
    return total, linear


NF4_BYTES_PER_WEIGHT = 0.5 + 1 / 64 + 4 / (64 * 256)  # 4 біти + 8-бітний масштаб на блок із 64 + fp32-масштаб на 256 блоків
rows = []
for name in MODELS:
    total, linear = count_parameters(name)
    free_memory()
    loaded = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16, device_map="auto")
    rows.append({"модель": name.split("/")[-1], "формат": "fp16", "параметрів, млрд": total / 1e9,
                 "очікувано, GB": total * 2 / 1e9, "виміряно, GB": torch.cuda.memory_allocated() / 1e9})
    del loaded
    free_memory()
    if name.endswith("4B"):
        loaded = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb_config, device_map="auto")
        rows.append({"модель": name.split("/")[-1], "формат": "NF4 (QLoRA)", "параметрів, млрд": total / 1e9,
                     "очікувано, GB": (linear * NF4_BYTES_PER_WEIGHT + (total - linear) * 2) / 1e9,
                     "виміряно, GB": torch.cuda.memory_allocated() / 1e9})
        del loaded
        free_memory()
memory_table = pd.DataFrame(rows)
memory_table["розбіжність, %"] = 100 * (memory_table["виміряно, GB"] / memory_table["очікувано, GB"] - 1)
memory_table.round(2)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

,модель,формат,"параметрів, млрд","очікувано, GB","виміряно, GB","розбіжність, %"
0,Qwen3-0.6B,fp16,0.60,1.19,1.21,1.48
1,Qwen3-1.7B,fp16,1.72,3.44,3.46,0.50
2,Qwen3-4B,fp16,4.02,8.04,8.06,0.21
3,Qwen3-4B,NF4 (QLoRA),4.02,2.65,2.70,1.66


In [30]:
fp16_4b = memory_table[(memory_table["формат"] == "fp16") & memory_table["модель"].str.endswith("4B")].iloc[0]
nf4_4b = memory_table[memory_table["формат"] == "NF4 (QLoRA)"].iloc[0]
print(f"fp16: виміряна памʼять відхиляється від формули «параметри × 2 байти» не більше ніж на "
      f"{memory_table[memory_table['формат'] == 'fp16']['розбіжність, %'].abs().max():.1f}%.")
print(f"NF4: {nf4_4b['виміряно, GB']:.2f} GB замість {fp16_4b['виміряно, GB']:.2f} GB — економія в {fp16_4b['виміряно, GB'] / nf4_4b['виміряно, GB']:.1f} рази, "
      "а не в 4: ембединги й нормалізації лишаються в fp16.")

fp16: виміряна памʼять відхиляється від формули «параметри × 2 байти» не більше ніж на 1.5%.
NF4: 2.70 GB замість 8.06 GB — економія в 3.0 рази, а не в 4: ембединги й нормалізації лишаються в fp16.


### Питання 24. Що робить рядок with torch.no_grad() у функції classify()?

**Відповідь.** `torch.no_grad()` — менеджер контексту, який **вимикає автоматичне диференціювання**: PyTorch не будує обчислювальний граф і не зберігає проміжні результати (активації всіх шарів), потрібні для зворотного проходу. Під час тренування вони потрібні, щоб порахувати градієнти, а в інференсі, коли модель лише відповідає, — ні. Тому в `no_grad`:
- **менше памʼяті** (не зберігаються активації, пропорційні кількості шарів × довжині × розміру батча);
- **швидше** (немає витрат на побудову графа);
- безпечніше: градієнти випадково не накопичуються.

Примітка: сам `model.generate` у Transformers теж виконується без градієнтів, тож у `classify_batch` явний `with torch.no_grad()` є додатковою страховкою й чітко показує намір. Різницю на звичайному прямому проході показує експеримент: той самий батч на Qwen3-0.6B з `no_grad` і без нього.

**Результат запуску** (батч 8 × 241 токен, Qwen3-0.6B, прямий прохід кістяка): без `no_grad` PyTorch будує граф (`grad_fn` є), пікова додаткова памʼять — 8,07 GB; з `no_grad` графа немає, пікова додаткова памʼять — лише 0,28 GB, тобто **у 29,2 раза менше**. Час проходу: 0,182 с проти 0,164 с, тобто приблизно на 10% швидше. Отже, головна перевага `no_grad` — памʼять, а виграш у часі невеликий.

In [31]:
import contextlib

small = AutoModelForCausalLM.from_pretrained(MODELS[0], dtype=torch.float16, device_map="auto").eval()
batch = tokenizer(["The quick brown fox jumps over the lazy dog. " * 24] * 8, return_tensors="pt").to(small.device)
print("Форма батча:", tuple(batch["input_ids"].shape))


def forward_cost(use_no_grad, repeats=3):
    """Пікова додаткова памʼять і середній час прямого проходу кістяка (без lm_head, щоб логіти не затьмарювали різницю)."""
    free_memory()
    torch.cuda.reset_peak_memory_stats()
    base = torch.cuda.memory_allocated()
    times = []
    for _ in range(repeats):
        torch.cuda.synchronize()
        start = time.perf_counter()
        with (torch.no_grad() if use_no_grad else contextlib.nullcontext()):
            hidden = small.model(**batch).last_hidden_state
        torch.cuda.synchronize()
        times.append(time.perf_counter() - start)
    info = {
        "граф обчислень збудовано (grad_fn)": hidden.grad_fn is not None,
        "пікова додаткова памʼять, GB": (torch.cuda.max_memory_allocated() - base) / 1e9,
        "час проходу, с": sum(times) / len(times),
    }
    del hidden
    return info


no_grad_table = pd.DataFrame({"без no_grad": forward_cost(False), "з no_grad": forward_cost(True)}).T
no_grad_table

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Форма батча: (8, 241)


,граф обчислень збудовано (grad_fn),"пікова додаткова памʼять, GB","час проходу, с"
без no_grad,True,8.066494,0.181838
з no_grad,False,0.276524,0.163633


In [32]:
ratio = no_grad_table.loc["без no_grad", "пікова додаткова памʼять, GB"] / max(no_grad_table.loc["з no_grad", "пікова додаткова памʼять, GB"], 1e-9)
print(f"Без no_grad пікова додаткова памʼять у {ratio:.1f} раза більша, бо зберігаються активації для зворотного проходу.")
del small
free_memory()

Без no_grad пікова додаткова памʼять у 29.2 раза більша, бо зберігаються активації для зворотного проходу.


### Питання 25. Для чого у classify() задано do_sample=False?

**Відповідь.** `do_sample=False` вмикає **жадібне декодування** (*greedy*): на кожному кроці береться найімовірніший токен. Результат **детермінований** (той самий запит → та сама відповідь) і для нашої задачі найточніший. Із `do_sample=True` наступний токен *випадково вибирається* за розподілом ймовірностей: навіть якщо правильна мітка має ймовірність 0,9, у ~10% випадків модель візьме іншу, а якщо випадково вибраним виявиться токен самої структури JSON, формат може зламатися. Крім того, випадковість унеможливлює чесне порівняння моделей і повторний запуск експерименту. Семплювання доречне для творчих задач (тексти, ідеї), де потрібна різноманітність, а не для класифікації з однією правильною відповіддю.

Експеримент: беремо найменшу донавчену модель (Qwen3-0.6B + збережений LoRA-адаптер) і класифікуємо 192 тестові звернення жадібно та із семплюванням (`temperature=1.0`, без обрізання розподілу), по два прогони кожен.

**Результат запуску** (192 тестові звернення, Qwen3-0.6B + адаптер):

| декодування | валідний JSON | category | intent | змінилося відповідей між двома прогонами |
|---|---|---|---|---|
| жадібне, прогін 1 | 100% | 100,0% | 99,0% | — |
| жадібне, прогін 2 | 100% | 100,0% | 99,0% | 0 з 192 |
| семплювання, прогін 1 | 100% | 100,0% | 95,3% | — |
| семплювання, прогін 2 | 100% | 99,5% | 95,3% | 12 з 192 |

На цій моделі семплювання **не зламало формат**: після SFT токени структури JSON мають імовірність майже 1, тож випадковість їх не зачіпає. Але воно зачепило значущі токени: точність intent впала з 99,0% до 95,3% (≈ 7 відповідей із 192), а повторний прогін змінив 12 відповідей із 192 (6%). Жадібне декодування дало ті самі відповіді в обох прогонах. Для задачі з однією правильною відповіддю семплювання — чиста втрата точності й відтворюваності.

In [33]:
from peft import PeftModel


def generate_answers(model, messages_batch, system_prompt, sample=False, temperature=1.0, max_new_tokens=60):
    """Те саме, що classify_batch, але з вибором між жадібним декодуванням і семплюванням."""
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": system_prompt}, {"role": "user", "content": message}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        for message in messages_batch
    ]
    tokenizer.padding_side = "left"
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    tokenizer.padding_side = "right"
    options = dict(do_sample=True, temperature=temperature, top_p=1.0, top_k=0) if sample else dict(do_sample=False)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id, **options)
    new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    return [answer.strip() for answer in tokenizer.batch_decode(new_tokens, skip_special_tokens=True)]


def score_answers(answers, targets):
    parsed = [parse(a) for a in answers]
    n = len(answers)
    return {
        "Валідний JSON, %": 100 * sum(p is not None for p in parsed) / n,
        "Точність category, %": 100 * sum(p is not None and p["category"] == t["category"] for p, t in zip(parsed, targets)) / n,
        "Точність intent, %": 100 * sum(p is not None and p["intent"] == t["intent"] for p, t in zip(parsed, targets)) / n,
    }


small = AutoModelForCausalLM.from_pretrained(MODELS[0], dtype=torch.float16, device_map="auto")
small = PeftModel.from_pretrained(small, "Qwen3-0.6B-lora-adapter").eval()

subset = test_dataset.select(range(192))
user_messages = [ex["messages"][1]["content"] for ex in subset]
targets = [json.loads(ex["messages"][2]["content"]) for ex in subset]


def run_all(sample):
    answers = []
    for start in range(0, len(user_messages), 32):
        answers += generate_answers(small, user_messages[start:start + 32], SHORT_PROMPT, sample=sample)
    return answers


runs = {
    "жадібно (do_sample=False), прогін 1": run_all(False),
    "жадібно (do_sample=False), прогін 2": run_all(False),
    "семплювання, прогін 1": run_all(True),
    "семплювання, прогін 2": run_all(True),
}
sampling_table = pd.DataFrame({name: score_answers(answers, targets) for name, answers in runs.items()}).T
sampling_table["відповідей, що змінилися відносно прогону 1"] = [
    0, sum(a != b for a, b in zip(runs["жадібно (do_sample=False), прогін 1"], runs["жадібно (do_sample=False), прогін 2"])),
    0, sum(a != b for a, b in zip(runs["семплювання, прогін 1"], runs["семплювання, прогін 2"])),
]
sampling_table.round(1)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

,"Валідний JSON, %","Точність category, %","Точність intent, %","відповідей, що змінилися відносно прогону 1"
"жадібно (do_sample=False), прогін 1",100.0,100.0,99.0,0
"жадібно (do_sample=False), прогін 2",100.0,100.0,99.0,0
"семплювання, прогін 1",100.0,100.0,95.3,0
"семплювання, прогін 2",100.0,99.5,95.3,12


In [34]:
greedy_row = sampling_table.iloc[0]
sample_row = sampling_table.iloc[2]
print(f"Жадібно: category {greedy_row['Точність category, %']:.1f}%, intent {greedy_row['Точність intent, %']:.1f}%; "
      f"повторний прогін змінив {int(sampling_table.iloc[1].iloc[-1])} відповідей із {len(subset)}.")
print(f"Семплювання: category {sample_row['Точність category, %']:.1f}%, intent {sample_row['Точність intent, %']:.1f}%; "
      f"повторний прогін змінив {int(sampling_table.iloc[3].iloc[-1])} відповідей із {len(subset)}.")
del small
free_memory()

Жадібно: category 100.0%, intent 99.0%; повторний прогін змінив 0 відповідей із 192.
Семплювання: category 100.0%, intent 95.3%; повторний прогін змінив 12 відповідей із 192.


### Питання 26. В демо 1 було натреновано класифікатор з базової моделі за допомогою заміни голови. В демо 3 в ході інференсу моделі також відбувається класифікація звернень. Чому не можна застосувати такий же підхід?

**Відповідь.** Строго кажучи, *можна*: для Qwen3 існує клас `Qwen3ForSequenceClassification` — той самий принцип «кістяк + нова лінійна голова» (код нижче це показує; лише замість `[CLS]` береться вектор **останнього токена**, бо в декодера з каузальною маскою тільки він «бачить» усі попередні токени). Але для задачі з демо 3 цей підхід гірший, і ось чому:

1. **Інший вихід.** У демо 1 відповідь — одне з двох чисел. У демо 3 потрібен **структурований JSON** із двома полями (`category` і `intent`) у заданому форматі. Генеративна модель видає його «з коробки», і формат легко розширити (додати поле = додати його в приклади). Голова ж фіксує **набір класів** в архітектурі: змінився перелік міток — потрібна нова голова й повне перенавчання.
2. **Голова не знає змісту міток.** Вона ініціалізується випадково, і для неї `cancel_order` — лише номер класу, тому потрібно достатньо прикладів на кожен із 27 класів. Генеративна модель уже знає слова «cancel» і «order», тож SFT лише привʼязує її знання до наших міток і зазвичай потребує менше прикладів (у цьому демо ми цього окремо не вимірювали).
3. **Універсальність.** Модель із класифікаційною головою вміє тільки класифікувати. Донавчена SFT-модель лишається генеративною: адаптер можна вимкнути, замінити іншим або поєднувати з іншими задачами тієї ж моделі.
4. **Мета демо.** Демо 1 показує *класичне* донавчання з новою головою на енкодері (BERT), демо 3 — SFT, у якому модель вчиться **генерувати** потрібну відповідь, зберігаючи свою архітектуру незмінною (див. вступ до демо 2).

Зворотний бік: для чистої класифікації з фіксованими класами голова на невеликому енкодері часто дешевша й швидша (один прохід замість генерації токен за токеном).

**Результат запуску.** Для Qwen3-0.6B клас `Qwen3ForSequenceClassification` справді створюється з тим самим кістяком, а замість `lm_head` має голову `Linear(1024 → 27)` без зсуву: 27 648 параметрів, що становить лише ≈ 0,005% моделі (596 077 568 параметрів). Кількість класів (27) зашита в розмір цього шару, тож інший перелік міток означав би нову голову й перенавчання.

In [35]:
from transformers import AutoModelForSequenceClassification

clf_config = AutoConfig.from_pretrained(MODELS[0])
clf_config.num_labels = len(intents)
with torch.device("meta"):
    clf_model = AutoModelForSequenceClassification.from_config(clf_config)
print("Клас:", type(clf_model).__name__)
print("Голова:", clf_model.score)
print(f"Параметрів голови: {clf_model.score.weight.numel():,} (= {clf_config.hidden_size} × {clf_config.num_labels}); "
      f"усього в моделі: {sum(p.numel() for p in clf_model.parameters()):,}")
print(f"Голова задає рівно {len(intents)} класів: інший перелік міток вимагатиме нової голови й перенавчання.")

Клас: Qwen3ForSequenceClassification
Голова: Linear(in_features=1024, out_features=27, bias=False)
Параметрів голови: 27,648 (= 1024 × 27); усього в моделі: 596,077,568
Голова задає рівно 27 класів: інший перелік міток вимагатиме нової голови й перенавчання.


### Питання 27. Порівняйте кількість токенів короткого й детального промпту (розділ 2). Як ця різниця вплине на вартість обробки 10 000 звернень на день?

**Відповідь.** Детальний промпт містить повний перелік категорій і намірів, тому він приблизно в 15 разів довший за короткий (166 проти 11 токенів), а його треба передавати **з кожним запитом** (модель без донавчання не знає наших міток). Після SFT мітки «зашиті» у ваги, тож вистачає короткого промпту. Нижче — точні числа й розрахунок для 10 000 звернень на день.

Вартість обробки пропорційна кількості токенів, які модель має прочитати (prefill). Для API з оплатою за токени це прямі гроші, для власного сервера — час GPU, памʼять KV-кешу й пропускна здатність. Ціни у таблиці — **умовні** (припущення для ілюстрації), реальні залежать від провайдера чи обладнання.

In [36]:
short_tokens = len(tokenizer(SHORT_PROMPT)["input_ids"])
long_tokens = len(tokenizer(LONG_PROMPT)["input_ids"])
print(f"Короткий промпт: {short_tokens} токенів; детальний: {long_tokens} токенів (у {long_tokens / short_tokens:.1f} раза більше)")


def request_tokens(system_prompt, user_message):
    """Токени всього запиту: системний промпт + звернення + службові токени шаблону розмови."""
    text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_message}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    return len(tokenizer(text)["input_ids"])


sample_messages = [ex["messages"][1]["content"] for ex in test_dataset.select(range(200))]
avg_short = sum(request_tokens(SHORT_PROMPT, m) for m in sample_messages) / len(sample_messages)
avg_long = sum(request_tokens(LONG_PROMPT, m) for m in sample_messages) / len(sample_messages)
print(f"Середній запит у токенах (з урахуванням звернення й шаблону): короткий {avg_short:.0f}, детальний {avg_long:.0f}")

REQUESTS_PER_DAY = 10_000
rows = []
for price in (0.10, 0.50, 3.00):  # умовна ціна, USD за 1 млн вхідних токенів
    rows.append({
        "ціна, $ за 1 млн токенів": price,
        "детальний промпт, $/день": avg_long * REQUESTS_PER_DAY / 1e6 * price,
        "короткий промпт, $/день": avg_short * REQUESTS_PER_DAY / 1e6 * price,
        "економія, $/день": (avg_long - avg_short) * REQUESTS_PER_DAY / 1e6 * price,
        "економія, $/міс (30 днів)": (avg_long - avg_short) * REQUESTS_PER_DAY * 30 / 1e6 * price,
    })
print(f"\nВхідних токенів на день: детальний {avg_long * REQUESTS_PER_DAY / 1e6:.2f} млн, короткий {avg_short * REQUESTS_PER_DAY / 1e6:.2f} млн "
      f"(різниця {(avg_long - avg_short) * REQUESTS_PER_DAY / 1e6:.2f} млн токенів щодня)")
pd.DataFrame(rows).round(2)

Короткий промпт: 11 токенів; детальний: 166 токенів (у 15.1 раза більше)
Середній запит у токенах (з урахуванням звернення й шаблону): короткий 38, детальний 193

Вхідних токенів на день: детальний 1.93 млн, короткий 0.38 млн (різниця 1.55 млн токенів щодня)


,"ціна, $ за 1 млн токенів","детальний промпт, $/день","короткий промпт, $/день","економія, $/день","економія, $/міс (30 днів)"
0,0.1,0.19,0.04,0.16,4.65
1,0.5,0.97,0.19,0.78,23.25
2,3.0,5.79,1.14,4.65,139.50


**Висновки.** Сам системний промпт скоротився з 166 до 11 токенів (у 15,1 раза), але весь запит разом зі зверненням клієнта й службовими токенами шаблону — із ≈ 193 до ≈ 38 токенів (у ≈ 5,1 раза), бо звернення й шаблон однакові в обох варіантах. На 10 000 звернень на день це **≈ 1,93 млн вхідних токенів проти ≈ 0,38 млн**: щодня обробляється на ≈ 1,55 млн токенів менше. За умовними тарифами таблиці економія становить від 0,16 $ на день (≈ 4,65 $ на місяць) при 0,10 $ за мільйон токенів до 4,65 $ на день (≈ 140 $ на місяць) при 3 $ за мільйон; для власного GPU це не гроші, а вища пропускна здатність і менша затримка кожного запиту.

Застереження: деякі провайдери та сервери (prefix caching) кешують однаковий початок запиту й беруть за нього менше, тому реальна економія може бути меншою; проте довгий промпт усе одно займає вікно контексту й обчислення уваги. До того ж економія токенів — не єдина вигода донавчання: модель точніше розрізняє мітки (наприклад, Qwen3-0.6B: точність intent 25,6% → 97,7%). Чистий JSON модель видавала й до донавчання з детальним промптом (100% відповідей), тож у цьому демо виграш — у точності та довжині промпту, а не в чистоті формату.

### Питання 28. Для чого в датасеті задано колонку chat_template_kwargs з enable_thinking=False? Що, на Вашу думку, сталося б із моделлю, якби її донавчали на цих прикладах у режимі /think?

**Відповідь.** Qwen3 — гібридна модель: прапорець `enable_thinking` у шаблоні розмови вмикає чи вимикає режим міркування. З `enable_thinking=False` шаблон дописує до **промпту для генерації** порожній блок `<think>\n\n</think>`, і модель одразу відповідає, без міркувань. Для нашої задачі це потрібно: JSON-класифікація не потребує довгих міркувань, які лише витрачали б токени, час і ризикували б зламати формат. Саме так ми генеруємо відповіді в `classify_batch` (`enable_thinking=False`).

Колонка `chat_template_kwargs` передає цей самий прапорець `SFTTrainer`, коли він застосовує шаблон до **тренувальних** діалогів. Принцип: формат тренування має збігатися з форматом інференсу.

**Важливе уточнення, яке показує клітинка нижче.** Для *завершеного* діалогу шаблон Qwen3 сам дописує порожній `<think>\n\n</think>` перед останньою відповіддю асистента, незалежно від `enable_thinking`: тренувальний текст для `False` і для `True` **однаковий** (вивід: «Тексти однакові: True»). Прапорець змінює лише промпт для генерації: з `False` він закінчується порожнім блоком, а з `True` — просто заголовком `<|im_start|>assistant`. Тому для цього шаблону колонка `chat_template_kwargs` не змінює тренувальний текст, а лише фіксує явно, у якому режимі ми працюємо.

**Що сталося б, якби донавчали в режимі `/think`** (`enable_thinking=True`) на цих прикладах, де відповіді асистента **не містять міркувань**:
1. Тренувальний текст, як щойно показано, не відрізнявся б: у кожному прикладі після `<|im_start|>assistant` стоїть порожній `<think>\n\n</think>`, а за ним одразу JSON. Тобто в будь-якому разі модель бачила б лише відповіді без міркувань.
2. Через це вона, найімовірніше, **розучилася б міркувати**, навіть коли ми вмикаємо `/think`: тренування привчило б її на заголовок асистента завжди відповідати порожнім `<think>` і одразу JSON. Нова поведінка перекриває стару (проблема на кшталт *catastrophic forgetting*), і на інших задачах, де міркування потрібні, якість упала б. Це лише очікування: такого експерименту ми не проводили. Те саме попередження наведено наприкінці демо 2 (на прикладі GSM8K).
3. Щоб зберегти здатність міркувати, у суміш для донавчання треба додавати приклади з блоком міркувань (так будують SmolTalk2: окремі частини `think` і `no_think`).

Клітинка нижче показує, як виглядає тренувальний текст за обох значень прапорця, що промпт інференсу є префіксом тренувального тексту (формати збігаються) і чим відрізняються промпти для генерації.

In [37]:
example = train_dataset[0]
messages_example = example["messages"]
print("chat_template_kwargs у прикладі:", example["chat_template_kwargs"])

text_no_think = tokenizer.apply_chat_template(messages_example, tokenize=False, **example["chat_template_kwargs"])
text_think = tokenizer.apply_chat_template(messages_example, tokenize=False, enable_thinking=True)
print("\n--- тренувальний текст, enable_thinking=False ---")
print(text_no_think)
print("--- тренувальний текст, enable_thinking=True ---")
print(text_think)
print("Тексти однакові:", text_no_think == text_think)

inference_prompt = tokenizer.apply_chat_template(messages_example[:2], tokenize=False, add_generation_prompt=True, enable_thinking=False)
print("\n--- промпт інференсу (system + user, enable_thinking=False, add_generation_prompt=True) ---")
print(inference_prompt)
print("Промпт інференсу є префіксом тренувального тексту:", text_no_think.startswith(inference_prompt))
print("Що модель має дописати після промпту:", repr(text_no_think[len(inference_prompt):]))

inference_prompt_think = tokenizer.apply_chat_template(messages_example[:2], tokenize=False, add_generation_prompt=True, enable_thinking=True)
print("\nПромпт інференсу з enable_thinking=True закінчується на:", repr(inference_prompt_think[-30:]))
print("Промпт інференсу з enable_thinking=False закінчується на:", repr(inference_prompt[-30:]))

chat_template_kwargs у прикладі: {'enable_thinking': False}

--- тренувальний текст, enable_thinking=False ---
<|im_start|>system
Classify the customer message. Reply with JSON only.<|im_end|>
<|im_start|>user
I have got to locate hte bills from {{Person Name}}<|im_end|>
<|im_start|>assistant
<think>

</think>

{"category": "INVOICE", "intent": "check_invoice"}<|im_end|>

--- тренувальний текст, enable_thinking=True ---
<|im_start|>system
Classify the customer message. Reply with JSON only.<|im_end|>
<|im_start|>user
I have got to locate hte bills from {{Person Name}}<|im_end|>
<|im_start|>assistant
<think>

</think>

{"category": "INVOICE", "intent": "check_invoice"}<|im_end|>

Тексти однакові: True

--- промпт інференсу (system + user, enable_thinking=False, add_generation_prompt=True) ---
<|im_start|>system
Classify the customer message. Reply with JSON only.<|im_end|>
<|im_start|>user
I have got to locate hte bills from {{Person Name}}<|im_end|>
<|im_start|>assistant
<think>

</thi

### Питання 29. Що означає параметр assistant_only_loss=True? На яких токенах модель вчилася б без нього і чим це погано для нашої задачі?

**Відповідь.** `assistant_only_loss=True` означає, що функція втрат (передбачення наступного токена) рахується **лише на токенах відповіді асистента**; для решти токенів (системне повідомлення, звернення клієнта, службові токени ролей) мітки замінюються на −100, і loss їх пропускає (padding пропускається завжди).

Без цього параметра loss за замовчуванням рахується на **всій послідовності**: модель вчилася б передбачати кожен токен, зокрема системний промпт, однаковий у кожному прикладі, і самі звернення клієнтів.

Чим це погано для нашої задачі:
1. **Сигнал розмивається.** Відповідь (JSON із двох міток) — це лише близько третини токенів прикладу (див. розрахунок нижче); решту loss вимагав би вивчити те, що нам не потрібно.
2. **Системний промпт повторюється в кожному прикладі,** тож модель швидко запамʼятала б його, і loss на цих токенах прямував би до нуля. Середній loss знизився б *«ні за що»* і перестав би відображати якість класифікації; validation loss стало б важче інтерпретувати.
3. **Ємність адаптера витрачалась би на моделювання *тексту звернень клієнтів***, хоча нам треба навчити модель їх **читати**, а не генерувати. Це розпорошує оновлення ваг між корисною й непотрібною поведінкою.
4. Ми вчимо поведінку «на звернення дай JSON», а не «пиши схожі звернення».

Тому для задач «запит → відповідь» вчать лише на відповідях. Нижче — яку частку токенів тренувальних прикладів складає відповідь асистента (оцінка за 300 прикладами; відповідь = усе після останнього заголовка `<|im_start|>assistant`, разом із порожнім блоком `<think>`).

**Результат запуску.** У середньому приклад має 54 токени, із них відповідь — 20 (37%); решта 34 токени (63%) — системний промпт, звернення клієнта й службові токени ролей. У ці 20 входять і порожній блок `<think>`, і `<|im_end|>`, тож частка самого JSON ще менша. Без `assistant_only_loss` loss рахувався б на всіх 100% токенів, і майже дві третини сигналу припадали б на текст, який модель не повинна генерувати.

In [38]:
marker = "<|im_start|>assistant\n"
rows = []
for ex in train_dataset.select(range(300)):
    full_text = tokenizer.apply_chat_template(ex["messages"], tokenize=False, **ex["chat_template_kwargs"])
    answer_start = full_text.rfind(marker) + len(marker)
    enc = tokenizer(full_text, return_offsets_mapping=True, add_special_tokens=False)
    total = len(enc["input_ids"])
    answer = sum(1 for start, _ in enc["offset_mapping"] if start >= answer_start)
    rows.append({"усього токенів": total, "токенів відповіді": answer})
token_stats = pd.DataFrame(rows)
share = token_stats["токенів відповіді"].sum() / token_stats["усього токенів"].sum()
print(f"У середньому {token_stats['усього токенів'].mean():.0f} токенів на приклад, із них відповідь — {token_stats['токенів відповіді'].mean():.0f} ({share:.0%}).")
print(f"З assistant_only_loss=True loss рахується на ≈ {share:.0%} токенів; без нього — на всіх 100%, зокрема на системному промпті й тексті звернення (≈ {1 - share:.0%}).")
token_stats.describe().round(1)

У середньому 54 токенів на приклад, із них відповідь — 20 (37%).
З assistant_only_loss=True loss рахується на ≈ 37% токенів; без нього — на всіх 100%, зокрема на системному промпті й тексті звернення (≈ 63%).


,усього токенів,токенів відповіді
count,300.0,300.0
mean,54.3,20.1
std,3.5,0.9
min,46.0,19.0
25%,52.0,19.0
50%,54.0,20.0
75%,56.0,21.0
max,67.0,22.0


### Питання 30. Порівняйте LoRA і QLoRA за пам'яттю, часом тренування і якістю (розділ 9). У яких випадках Ви обрали б QLoRA?

**Відповідь.** LoRA і QLoRA тренують **ті самі адаптери** з однаковими гіперпараметрами. Різниця в тому, як зберігаються *заморожені* ваги базової моделі: fp16 (LoRA) чи 4-бітний NF4 (QLoRA). Таблиця нижче порівнює Qwen3-4B за підсумковими даними розділу 9.

**Результат запуску** (Qwen3-4B, тестовий набір із 1000 прикладів):
- **памʼять:** пік на GPU — 8,75 GB у LoRA проти 3,67 GB у QLoRA, тобто QLoRA використовує 42% памʼяті LoRA (економія 58%). Це менше за економію на самих вагах (8,06 → 2,70 GB, 67%, питання 23), бо пік містить ще активації й стани оптимізатора для адаптерів, які в обох методах однакові;
- **час:** тренування тривало 3,93 хв у LoRA і 5,13 хв у QLoRA, тобто QLoRA повільніша у 1,30 раза: перед кожним множенням 4-бітні ваги деквантизуються;
- **якість:** точність category — 100,0% проти 99,9%, intent — 97,8% проти 97,8%. Різниця в одну помилку на 1000 прикладів (0,1 п.п.) — у межах шуму, тож квантизація базових ваг тут нічого помітно не коштувала.

In [39]:
lora_vs_qlora = pd.DataFrame({"LoRA (fp16)": summary["Qwen3-4B LoRA"], "QLoRA (NF4)": summary["Qwen3-4B QLoRA"]})
lora_vs_qlora["QLoRA / LoRA"] = lora_vs_qlora["QLoRA (NF4)"] / lora_vs_qlora["LoRA (fp16)"]
display(lora_vs_qlora.round(3))

mem_ratio = lora_vs_qlora.loc["Пікова памʼять GPU, GB", "QLoRA / LoRA"]
time_ratio = lora_vs_qlora.loc["Тривалість, хв", "QLoRA / LoRA"]
cat_diff = lora_vs_qlora.loc["Точність category, %", "QLoRA (NF4)"] - lora_vs_qlora.loc["Точність category, %", "LoRA (fp16)"]
int_diff = lora_vs_qlora.loc["Точність intent, %", "QLoRA (NF4)"] - lora_vs_qlora.loc["Точність intent, %", "LoRA (fp16)"]
print(f"Памʼять: QLoRA використовує {mem_ratio:.0%} від памʼяті LoRA (економія {1 - mem_ratio:.0%}).")
print(f"Час тренування: QLoRA {'довше' if time_ratio > 1 else 'коротше'} у {max(time_ratio, 1 / time_ratio):.2f} раза.")
print(f"Якість: різниця точності category {cat_diff:+.1f} п.п., intent {int_diff:+.1f} п.п. на тестовому наборі з {len(test_dataset)} прикладів "
      f"(одна помилка = {100 / len(test_dataset):.1f} п.п.).")

,LoRA (fp16),QLoRA (NF4),QLoRA / LoRA
"Чистий JSON, %",100.000,100.000,1.000
"Валідний JSON, %",100.000,100.000,1.000
"Точність category, %",100.000,99.900,0.999
"Точність intent, %",97.800,97.800,1.000
"Тривалість, хв",3.933,5.129,1.304
"Пікова памʼять GPU, GB",8.746,3.666,0.419
Validation loss,0.003,0.005,1.466
"Адаптер, MB",132.188,NaN,NaN


Памʼять: QLoRA використовує 42% від памʼяті LoRA (економія 58%).
Час тренування: QLoRA довше у 1.30 раза.
Якість: різниця точності category -0.1 п.п., intent +0.0 п.п. на тестовому наборі з 1000 прикладів (одна помилка = 0.1 п.п.).


**Коли обрав би QLoRA:**
- коли **модель не вміщується** у памʼять GPU в fp16 навіть з LoRA: наприклад, 7–8B-модель на T4 з 15 GB (самі ваги в fp16 займають ≈ 16 GB), а в NF4 — ≈ 5–6 GB; так само 30B+ на одній картці на 24–48 GB;
- коли треба донавчити **більшу** модель на дешевому чи споживчому залізі (безкоштовний Colab, домашня відеокарта), і додатковий час тренування прийнятний;
- коли виграш у якості від більшої моделі важливіший за швидкість експерименту.

**Коли обрав би звичайну LoRA:** коли модель із запасом вміщується в fp16 (як 0.6B і 1.7B на T4, а за наявності памʼяті — і 4B), бо тренування швидше (немає деквантизації ваг перед кожним множенням) і немає ризику втрат точності від квантизації; для найвищої якості на фінальному запуску.

**Висновок:** QLoRA — спосіб розмінювати час (і трохи точності) на памʼять. Якщо памʼяті вистачає, переваг у ній немає; якщо ні — вона дозволяє зробити те, що без неї неможливо.